# PyTorch FFNN with BatchNorm and Dropout
This notebook trains a Feedforward Neural Network on the California Housing dataset using Batch Normalization and Dropout.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from personalized_batchnorm import PersonalizedBatchNorm1d
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Data Preparation
X, y = fetch_california_housing(return_X_y=True)

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = X_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

early_stopping_patience = 50  # Number of epochs to wait for improvement before stopping

# 2. Model Architecture (BatchNorm1d)
class CaliforniaHousingFFNN(nn.Module):
    def __init__(self, input_dim, hidden_dim1=64, hidden_dim2=32, dropout_rate=0.1):
        super(CaliforniaHousingFFNN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            PersonalizedBatchNorm1d(hidden_dim1), # Normalizes across the batch
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim1, hidden_dim2),
            PersonalizedBatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim2, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize model, loss, and optimizer
input_size = X_train.shape[1]
model = CaliforniaHousingFFNN(input_dim=input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop
epochs = 500
batch_size = 32
dataset_size = len(X_train_t)

best_loss = float('inf')
last_improvement_epoch = 0

for epoch in range(epochs):
    model.train() 
    
    permutation = torch.randperm(dataset_size)
    epoch_loss = 0.0
    
    
    for i in range(0, dataset_size, batch_size):
        indices = permutation[i:i + batch_size]
        batch_X, batch_y = X_train_t[indices], y_train_t[indices]
        
        if len(batch_X) == 1:
            continue
            
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(X_train_t):.4f}")

    
    last_improvement_epoch = epoch if epoch_loss / len(X_train_t) < best_loss else last_improvement_epoch
    best_loss = min(best_loss, epoch_loss / len(X_train_t))

    if last_improvement_epoch < epoch - early_stopping_patience:
        print(f"Early stopping at epoch {epoch+1} due to no improvement in loss for {early_stopping_patience} epochs.")
        break


# 4. Evaluation
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_loss = criterion(test_preds, y_test_t)
    print(f"\nFinal Test MSE: {test_loss.item():.4f}")


KeyboardInterrupt: 